In [ ]:
# ===== NB2: THE BIG LADDER (pure CPU, all day, uphill-then-downhill) ====
# Iterated orbit-floor descent over all 124 classes: rungs 20, beam 32.
# Chains climb before dropping; any *** LEAD mu<= line could be an
# outright stable solve (mu<=12 criterion) — verification bar applies.
REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "research/w5/stable-ac-escape"   # must match the actual git branch
REPO_DIR = "ACSolverX"
UPDATE_REPO = True
DRIVE_DIR = "/content/drive/MyDrive/acsolverx_results/stable_ac_escape"
RUNGS, BEAM, JOBS = 20, 32, 8


In [ ]:
# ==================== SETUP (clone / pull / install / drive) ==============
import glob, os, shutil, subprocess, sys, time

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy pyyaml")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
else:
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
for m in [k for k in sys.modules if k == "experiments" or k.startswith("experiments.")]:
    del sys.modules[m]
print("repo root:", REPO_ROOT)

def _sync(src, dst):
    # size-monotonic whole-file copy: append-only jsonls only ever grow
    if os.path.exists(src) and (not os.path.exists(dst)
                                or os.path.getsize(dst) < os.path.getsize(src)):
        tmp = dst + ".tmp"
        shutil.copyfile(src, tmp)
        os.replace(tmp, dst)

def seed_from_drive(local_dir):
    """Fresh VM: pull this run's jsonls back from Drive so resume continues."""
    if not (IN_COLAB and os.path.isdir(DRIVE_DIR)):
        return
    os.makedirs(local_dir, exist_ok=True)
    for src in glob.glob(os.path.join(DRIVE_DIR, "*.jsonl")):
        _sync(src, os.path.join(local_dir, os.path.basename(src)))

def mirror_to_drive(local_dir):
    if not (IN_COLAB and os.path.isdir(DRIVE_DIR)):
        return
    for src in glob.glob(os.path.join(local_dir, "*.jsonl")):
        _sync(src, os.path.join(DRIVE_DIR, os.path.basename(src)))

def run_cli_mirrored(cmd, local_dir, every_s=180):
    """Run a resume-safe CLI as a subprocess, mirroring jsonls to Drive every
    few minutes from THIS main thread (never a background thread)."""
    seed_from_drive(local_dir)
    env = dict(os.environ, ACSOLVERX_ALLOW_BIG="1")
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=env)
    last = time.monotonic()
    for line in p.stdout:
        print(line, end="")
        if time.monotonic() - last > every_s:
            mirror_to_drive(local_dir)
            last = time.monotonic()
    p.wait()
    mirror_to_drive(local_dir)
    print("exit:", p.returncode)


In [ ]:
# ==================== RUN =================================================
run_cli_mirrored(
    f"python3 -m experiments.stable_ac.cov.mu_ladder --rungs {RUNGS} "
    f"--beam {BEAM} --jobs {JOBS}",
    os.path.join(REPO_ROOT, "results/stable_ac/mu_scan"))
